# Experimentos SciBERT/SPECTER con datos limpios (issue #104)

Orquestador **reanudable** para correr de noche. La lógica vive en `clasificacion_funcion.{seleccion_datos,
experimento_104, registro_mlflow}`. Todo se registra en MLflow (`clasificacion-funcion-104`).

> **Provisional**: no hay Test Gold humano. Las métricas son contra `val` (V fija), `dev_haiku` y `ref_gpt`.
> **Regla anti-fuga**: se excluyen los citantes de 2014 y 2015, el manifiesto v2, TG2700, taller y piloto.

Primero `DRY = True` (64 casos, 1 época) para validar el flujo; luego `DRY = False`.

In [1]:
import json
import sys
import time
import traceback
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "packages/clasificacion-funcion/src"))
sys.path.insert(0, str(REPO / "packages/auditoria-corpus/src"))
sys.path.insert(0, str(REPO / "packages/recuperacion-densa/src"))
sys.path.insert(0, str(REPO))  # `src.tracking` y `proyecto_de_grado` (namespace desde la raiz)

import mlflow

from clasificacion_funcion import adaptador_proyecto_grado as ad
from clasificacion_funcion import clasificador_torch as ct
from clasificacion_funcion import experimento_104 as ex
from clasificacion_funcion import registro_mlflow as rm
from clasificacion_funcion import seleccion_datos as sd

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DRY = False          # True: 64 casos y 1 epoca por run, experimento y carpetas aparte
HORAS_MAX = 7.0     # se omiten las fases restantes al superarlo
SEMILLAS = (42, 1234, 2026)

SUFIJO = "-dry" if DRY else ""
ARTIFACTS = REPO / f"proyecto_de_grado/artifacts/experimentos_104{'_dry' if DRY else ''}"
EVIDENCIAS = REPO / f"proyecto_de_grado/docs/evidencias_experimentos_104{'_dry' if DRY else ''}"
CORRIDAS = ARTIFACTS / "corridas"
T0 = time.time()
FALLAS = []

rm.configurar(rm.EXPERIMENTO + SUFIJO)
print("tracking:", mlflow.get_tracking_uri(), "| DRY =", DRY)


def hay_tiempo() -> bool:
    return (time.time() - T0) / 3600 < HORAS_MAX


def correr(nombre, fn, *a, **kw):
    """Ejecuta un run sin detener el notebook: la falla queda FAILED en MLflow y en FALLAS."""
    if not hay_tiempo():
        print(f"[omitido por HORAS_MAX] {nombre}")
        return None
    t = time.time()
    try:
        run_id = fn(nombre, *a, **kw)
        print(f"[ok {time.time() - t:6.0f}s] {nombre} -> {run_id}")
        return run_id
    except Exception:
        FALLAS.append(nombre)
        print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
        return None


def puntaje(run_id):
    """Macro-F1 medio en dev_haiku y ref_gpt (criterio de eleccion, registrado en MLflow)."""
    if not run_id:
        return -1.0
    m = mlflow.get_run(run_id).data.metrics
    return sum(m.get(k, 0.0) for k in ("dev_haiku_f1_macro", "ref_gpt_f1_macro")) / 2

tracking: sqlite:////home/carlos/microproyecto-local-citation/mlflow.db | DRY = False


## Fase 0 — Datos limpios (P, V, D1–D6), asserts de fuga y auditoría de fuentes

In [3]:
FUENTES = ad.cargar_fuentes_104()
PAPERS = ad.cargar_papers()
RES = sd.construir_todo(FUENTES)          # lanza ValueError si hay cualquier fuga
V, VARIANTES = RES["v"], RES["variantes"]
MANIFIESTO = sd.escribir_artefactos(RES, ARTIFACTS / "datos", EVIDENCIAS, {
    "qlora_18000_v2": ad.PREDS_QLORA[0], "jurado": ad.JURADO, "referencia_gpt": ad.REFERENCIA_GPT,
    "haiku_2700": ad.HAIKU_2700, "split_v2": ad.SPLIT_V2})
SHA_DATOS = sd.sha_archivo(EVIDENCIAS / "manifiesto_datos.json")
AUDITORIA = sd.auditar_fuentes(FUENTES)
(EVIDENCIAS / "auditoria_fuentes.json").write_text(json.dumps(AUDITORIA, ensure_ascii=False, indent=2), encoding="utf-8")

print("pool:", RES["info_pool"])
print("fugas:", RES["fugas"]["total_fugas"], "| citas de anios excluidos:", RES["fugas"]["citas_de_anios_excluidos"])
for n, c in MANIFIESTO["conteos"].items():
    print(f"{n:11s} n={MANIFIESTO['n'][n]:6d}", list(c.values()))

with rm.abrir_corrida("auditoria-datos", {"fase": 0, "sha_datos": SHA_DATOS, "dry": DRY}, {"manifiesto_datos_sha": SHA_DATOS}) as run:
    if run is not None:
        mlflow.log_metric("fugas_total", RES["fugas"]["total_fugas"])
        mlflow.log_metrics({f"n_{n}": v for n, v in MANIFIESTO["n"].items()})
        mlflow.log_dict(MANIFIESTO, "manifiesto_datos.json")
        mlflow.log_dict(AUDITORIA, "auditoria_fuentes.json")
        for par, v in AUDITORIA["acuerdo_pares"].items():
            if v["acuerdo"] is not None:
                mlflow.log_metric("acuerdo_" + par.replace("~", "_vs_"), v["acuerdo"])
AUDITORIA["acuerdo_pares"]

pool: {'pool': 40487, 'descartes': {'sin_etiqueta_confiable': 3398, 'qlora_abstiene': 664, 'haiku_abstiene': 58, 'id_vetado': 12, 'anio_excluido': 19095}}
fugas: 0 | citas de anios excluidos: 0
qlora_p50   n= 13332 [2000, 1212, 1613, 2000, 2000, 656, 2000, 970, 881]
qlora_p80   n=  9826 [2000, 227, 568, 2000, 2000, 314, 1414, 613, 690]
jurado      n=  8887 [2000, 683, 379, 1433, 2000, 771, 1386, 85, 150]
hibrido     n= 13719 [2000, 1449, 1505, 2000, 2000, 1031, 1998, 895, 841]
natural     n= 13331 [5354, 448, 600, 1410, 3698, 235, 905, 358, 323]
haiku_solo  n=  2180 [844, 76, 120, 262, 586, 48, 137, 60, 47]
validacion  n=  3379 [1307, 136, 120, 441, 899, 107, 262, 59, 48]


{'qlora~jurado': {'n': 14380, 'acuerdo': 0.7075104311543811},
 'qlora~haiku': {'n': 0, 'acuerdo': None},
 'qlora~gpt': {'n': 199, 'acuerdo': 0.6984924623115578},
 'jurado~haiku': {'n': 0, 'acuerdo': None},
 'jurado~gpt': {'n': 163, 'acuerdo': 0.7423312883435583},
 'haiku~gpt': {'n': 0, 'acuerdo': None}}

In [4]:
AUDITORIA["precision_qlora_por_p"]

{'qlora_vs_haiku_p[0.5,0.6)': {'n': 0, 'precision': None},
 'qlora_vs_haiku_p[0.6,0.7)': {'n': 0, 'precision': None},
 'qlora_vs_haiku_p[0.7,0.8)': {'n': 0, 'precision': None},
 'qlora_vs_haiku_p[0.8,0.9)': {'n': 0, 'precision': None},
 'qlora_vs_haiku_p[0.9,0.95)': {'n': 0, 'precision': None},
 'qlora_vs_haiku_p[0.95,1.0)': {'n': 0, 'precision': None},
 'qlora_vs_gpt_p[0.5,0.6)': {'n': 15, 'precision': 0.6666666666666666},
 'qlora_vs_gpt_p[0.6,0.7)': {'n': 25, 'precision': 0.28},
 'qlora_vs_gpt_p[0.7,0.8)': {'n': 16, 'precision': 0.75},
 'qlora_vs_gpt_p[0.8,0.9)': {'n': 32, 'precision': 0.5625},
 'qlora_vs_gpt_p[0.9,0.95)': {'n': 13, 'precision': 0.6923076923076923},
 'qlora_vs_gpt_p[0.95,1.0)': {'n': 82, 'precision': 0.9512195121951219}}

## Backfill de las 18 corridas de #103 (sin reentrenar)

In [5]:
if not DRY:
    A4 = REPO / "proyecto_de_grado/artifacts/clasificacion_a4/corridas"
    print(len(rm.registrar_corridas_a4(A4, rm.EXPERIMENTO)), "runs de #103 nuevos")

0 runs de #103 nuevos


## Fase 1 — Línea base TF-IDF + LR sobre D1

In [6]:
COMUN = {"v_sha": MANIFIESTO["sha256_ids"]["validacion"], "datos_sha": SHA_DATOS}
D1 = VARIANTES["qlora_p50"]
correr("tfidf-lr-d1", ex.linea_base_tfidf, {**COMUN, "variante": "qlora_p50"}, D1, V, FUENTES, PAPERS, dry=DRY)

[ok     37s] tfidf-lr-d1 -> 2b1a3594897c4b04921e916ee01e7078


'2b1a3594897c4b04921e916ee01e7078'

## Fase 2 — SciBERT-A (semilla 42) en D1…D6

In [7]:
def sci(nombre, variante, semilla=42, modo="duro", pesos_clase=True, encoder="scibert"):
    cfg = {**COMUN, "variante": variante, "ids_train_sha": MANIFIESTO["sha256_ids"][variante]}
    hp = ct.Hiperparametros(pesos_clase=pesos_clase)
    return correr(nombre, ex.entrenar_y_registrar, cfg, VARIANTES[variante], V, FUENTES, PAPERS, CORRIDAS,
                  encoder=encoder, semilla=semilla, modo=modo, hp=hp, dry=DRY, manifiesto_datos_sha=SHA_DATOS)


FASE2 = {v: sci(f"scibert-A-{v}-s42", v) for v in sd.VARIANTES}
FASE2_PUNTAJE = {v: puntaje(r) for v, r in FASE2.items()}
MEJOR_DATOS = max(FASE2_PUNTAJE, key=FASE2_PUNTAJE.get)
print(FASE2_PUNTAJE, "-> mejor variante de datos:", MEJOR_DATOS)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22927.25it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1349s] scibert-A-qlora_p50-s42 -> 9e29bd16e11f44ab9fced57e397c2883


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 54090.24it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    923s] scibert-A-qlora_p80-s42 -> eb9615c94670469790ee9bc8c879845c


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23254.95it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    713s] scibert-A-jurado-s42 -> d4ec92f99dad4a798563b83e92ce4f6e


... (2 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 54188.57it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1214s] scibert-A-natural-s42 -> f5ed045922b744cdb5a1fba68a279be5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32562.19it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    273s] scibert-A-haiku_solo-s42 -> 350f9c9d9f4e466b9112da31eeff55e2
{'qlora_p50': 0.5310791386826184, 'qlora_p80': 0.5358747017129946, 'jurado': 0.5147504155951963, 'hibrido': 0.5061555239900903, 'natural': 0.5402475380666056, 'haiku_solo': 0.500735978572836} -> mejor variante de datos: natural


## Fase 3 — Sobre la mejor variante: pérdida suave (KD), ponderada por p, sin pesos de clase

In [8]:
FASE3 = {"duro": FASE2[MEJOR_DATOS],
         "kd": sci(f"scibert-A-{MEJOR_DATOS}-kd-s42", MEJOR_DATOS, modo="kd"),
         "ponderado": sci(f"scibert-A-{MEJOR_DATOS}-ponderado-s42", MEJOR_DATOS, modo="ponderado"),
         "sin_pesos": sci(f"scibert-A-{MEJOR_DATOS}-sinpesos-s42", MEJOR_DATOS, pesos_clase=False)}
FASE3_PUNTAJE = {k: puntaje(r) for k, r in FASE3.items()}
MEJOR_CONFIG = max(FASE3_PUNTAJE, key=FASE3_PUNTAJE.get)
CFG = {"duro": {}, "kd": {"modo": "kd"}, "ponderado": {"modo": "ponderado"},
       "sin_pesos": {"pesos_clase": False}}[MEJOR_CONFIG]
SUF = {"duro": "", "kd": "-kd", "ponderado": "-ponderado", "sin_pesos": "-sinpesos"}[MEJOR_CONFIG]
print(FASE3_PUNTAJE, "-> mejor configuracion:", MEJOR_CONFIG)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16931.39it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1224s] scibert-A-natural-kd-s42 -> 244b433e16e04b56a06ddf636d414e24


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42381.77it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1220s] scibert-A-natural-ponderado-s42 -> 930c90ac35b94bb1b30eadd499e3e473


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27205.56it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1224s] scibert-A-natural-sinpesos-s42 -> e4f0170b7e964b7c983b723206a4bc7e
{'duro': 0.5402475380666056, 'kd': 0.5402475380666056, 'ponderado': 0.5402475380666056, 'sin_pesos': 0.561807564882767} -> mejor configuracion: sin_pesos


## Fase 4 — SPECTER-A con la mejor configuración

In [9]:
R_SPECTER = sci(f"specter-A-{MEJOR_DATOS}{SUF}-s42", MEJOR_DATOS, encoder="specter", **CFG)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39848.49it/s]


[ok   1224s] specter-A-natural-sinpesos-s42 -> df422e2520db4341af7a8b69e7bddb6d


## Fase 5 — Mejor configuración de SciBERT con otras semillas

In [10]:
R_SEMILLAS = {42: FASE3[MEJOR_CONFIG]}
for s in SEMILLAS[1:]:
    R_SEMILLAS[s] = sci(f"scibert-A-{MEJOR_DATOS}{SUF}-s{s}", MEJOR_DATOS, semilla=s, **CFG)
import statistics
for metrica in ("val_f1_macro", "dev_haiku_f1_macro", "ref_gpt_f1_macro"):
    xs = [mlflow.get_run(r).data.metrics.get(metrica) for r in R_SEMILLAS.values() if r]
    xs = [x for x in xs if x is not None]
    if len(xs) > 1:
        print(f"{metrica}: {statistics.mean(xs):.4f} +- {statistics.stdev(xs):.4f} (n={len(xs)})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20004.47it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1213s] scibert-A-natural-sinpesos-s1234 -> a3efed3ec1bc4166a50156c0a543b38c


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 62214.26it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1204s] scibert-A-natural-sinpesos-s2026 -> 22da6f29c7d2487c8c272b2da354874f
val_f1_macro: 0.6882 +- 0.0067 (n=3)
dev_haiku_f1_macro: 0.5982 +- 0.0203 (n=3)
ref_gpt_f1_macro: 0.4946 +- 0.0365 (n=3)


## Fase 6 — Smoke test de inferencia (SciBERT y SPECTER)

In [11]:
SMOKE = []
for encoder, nombre in (("scibert", f"scibert-A-{MEJOR_DATOS}{SUF}-s42"), ("specter", f"specter-A-{MEJOR_DATOS}{SUF}-s42")):
    carpeta = CORRIDAS / nombre
    if not (carpeta / "model.safetensors").is_file():
        print("sin checkpoint:", nombre)
        continue
    try:
        r = ex.smoke_inferencia(carpeta, encoder, V, FUENTES, PAPERS)
    except Exception:
        FALLAS.append(f"smoke-{encoder}")
        r = {"encoder": encoder, "ok": False, "error": traceback.format_exc()}
    SMOKE.append(r)
    print(r)
(EVIDENCIAS / "smoke_inferencia.json").write_text(json.dumps(SMOKE, ensure_ascii=False, indent=2), encoding="utf-8")
with rm.abrir_corrida("smoke-inferencia", {"fase": 6, "mejor": f"{MEJOR_DATOS}{SUF}", "dry": DRY}) as run:
    if run is not None:
        for r in SMOKE:
            mlflow.log_metric(f"smoke_ok_{r['encoder']}", int(r["ok"]))
            if "latencia_ms" in r:
                mlflow.log_metric(f"smoke_latencia_ms_{r['encoder']}", r["latencia_ms"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27574.96it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'run': 'scibert-A-natural-sinpesos-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.538499843737554, 'checkpoint_sha256': '0591ced42ce112e1d2c78178a3abb732ebd0d5d7ffd95095f8577b153e4bec89'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16490.50it/s]


{'run': 'specter-A-natural-sinpesos-s42', 'encoder': 'specter', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.600946937477147, 'checkpoint_sha256': '0a41d05ad7a6b74b3da45a4f57a22741c95c18d4b50cb8d6fb0e4e92e44e6bae'}


## Fase 7 — Entrada B (opcional)

Solo corre si existen las entradas B preparadas (`artifacts/experimentos_104/entrada_b/`). La preparación usa red y
GPU de día (ver runbook); de noche no se usa la red. Si no están, la fase queda registrada como omitida.

In [12]:
ENTRADA_B = ARTIFACTS / "entrada_b"
with rm.abrir_corrida("fase7-entrada-b", {"fase": 7, "dry": DRY, "disponible": ENTRADA_B.is_dir()}) as run:
    if run is not None:
        mlflow.set_tag("estado_fase", "omitida_sin_entradas_b" if not ENTRADA_B.is_dir() else "pendiente_implementar")
print("entrada B disponible:", ENTRADA_B.is_dir())

entrada B disponible: False


## Fase 8 — D7 `sonnet` (opcional)

Solo si existe `etiquetas_sonnet.csv` (anotación con Sonnet).

In [13]:
SONNET = REPO / "proyecto_de_grado/artifacts/anotacion_sonnet/etiquetas_sonnet.csv"
with rm.abrir_corrida("fase8-d7-sonnet", {"fase": 8, "dry": DRY, "disponible": SONNET.is_file()}) as run:
    if run is not None:
        mlflow.set_tag("estado_fase", "omitida_sin_etiquetas_sonnet" if not SONNET.is_file() else "pendiente_implementar")
print("etiquetas Sonnet disponibles:", SONNET.is_file())

etiquetas Sonnet disponibles: False


## Fase 9 — Manifiesto de experimentos

In [14]:
DF = rm.exportar_manifiesto(EVIDENCIAS, rm.EXPERIMENTO + SUFIJO)
print(len(DF), "runs exportados a", EVIDENCIAS)
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")
cols = [c for c in ("tags.mlflow.runName", "status", "metrics.val_f1_macro", "metrics.dev_haiku_f1_macro", "metrics.ref_gpt_f1_macro") if c in DF]
DF[cols].sort_values("metrics.val_f1_macro", ascending=False) if "metrics.val_f1_macro" in cols else DF[cols]

35 runs exportados a /home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104
fallas: ninguna | 3.58 h


,tags.mlflow.runName,status,metrics.val_f1_macro,metrics.dev_haiku_f1_macro,metrics.ref_gpt_f1_macro
3,scibert-A-natural-sinpesos-s2026-20261111-073914,FINISHED,0.695820,0.578049,0.524872
8,scibert-A-natural-kd-s42-20261111-055729,FINISHED,0.694479,0.614898,0.465598
7,scibert-A-natural-ponderado-s42-20261111-061753,FINISHED,0.694479,0.614898,0.465598
10,scibert-A-natural-s42-20261111-053242,FINISHED,0.694479,0.614898,0.465598
6,scibert-A-natural-sinpesos-s42-20261111-063813,FINISHED,0.685169,0.618733,0.504882
4,scibert-A-natural-sinpesos-s1234-20261111-071900,FINISHED,0.683540,0.597873,0.453994
5,specter-A-natural-sinpesos-s42-20261111-065837,FINISHED,0.679131,0.587021,0.470592
13,scibert-A-qlora_p80-s42-20261111-044734,FINISHED,0.655693,0.587654,0.484096
14,scibert-A-qlora_p50-s42-20261111-042505,FINISHED,0.649771,0.554581,0.507578
11,scibert-A-hibrido-s42-20261111-051449,FINISHED,0.638885,0.559591,0.452721
